# 학생 기반 분할 ML 분류 실험

이 노트북은 55명의 학생을 비율별로 훈련/테스트에 **완전 분리**하여, ML 모델이 **학습하지 않은 새로운 학생**의 피드백을 얼마나 정확하게 분류하는지 검증합니다.

**실험 설계**:
- **5가지 비율**: 80:20, 70:30, 60:40, 50:50, 40:60 (훈련 학생 수를 점차 줄여가며 성능 변화 관찰)
- **3가지 시드(seed)**: 각 비율에서 서로 다른 학생 배정으로 결과의 안정성 확인
- **24개 ML 모델**: 9개 개별 + 15개 앙상블
- **총 360회 실험**: 15 분할 x 24 모델

**핵심 차이점**: `01_data_exploration.ipynb`의 80:20 분할은 데이터 포인트 단위의 무작위 분할이지만, 이 노트북은 **학생 단위** 분할로 동일 학생의 데이터가 양쪽에 섞이지 않습니다.

**테스트 예측 방법**: 10-Fold CV에서 학습된 10개 모델의 예측을 **다수결 투표(majority voting)**로 결합합니다.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import os
from datetime import datetime
from copy import deepcopy
import json
from tqdm.auto import tqdm
from scipy import stats  # mode 함수: 다수결 투표에 사용

warnings.filterwarnings('ignore')

# 형태소 분석기 (한국어 텍스트 토큰화)
from kiwipiepy import Kiwi
# 텍스트 벡터화
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
# 개별 분류 알고리즘
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
# 앙상블 분류 알고리즘
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,       # 투표 기반 앙상블
    StackingClassifier       # 스태킹 기반 앙상블
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
# 교차 검증 및 평가 지표
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

print(f"실험 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("라이브러리 로드 완료!")

In [ ]:
DATA_PATH = Path("../data/feedback_data.xlsx")
OUTPUT_DIR = Path("./student_split_results")  # 실험 결과 저장 디렉토리
PRED_DIR = OUTPUT_DIR / 'predictions'          # 예측값 저장 디렉토리
OUTPUT_DIR.mkdir(exist_ok=True)
PRED_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42    # CV 내부에서 사용하는 시드 (분할 시드와 별개)
N_FOLDS = 10        # 10-Fold 교차 검증

# 학생 분할 비율: (훈련%, 테스트%)
SPLIT_RATIOS = [
    (80, 20),   # 가장 많은 훈련 데이터
    (70, 30),
    (60, 40),
    (50, 50),   # 동일 비율
    (40, 60),   # 가장 적은 훈련 데이터
]

# 3개의 서로 다른 시드로 학생 배정의 다양성 확보
SPLIT_SEEDS = [42, 123, 456]

# === ML 모델 설정 (02_structural_split.ipynb과 동일) ===
INDIVIDUAL_MODELS = ['LR', 'SVM-L', 'SVM-RBF', 'CNB', 'DT', 'RF', 'GB', 'XGB', 'LGBM']

ENSEMBLE_COMBOS = {
    'C1_Optimal5': ['SVM-L', 'CNB', 'DT', 'GB', 'XGB'],
    'C2_Alt5':     ['LR', 'SVM-L', 'DT', 'GB', 'XGB'],
    'C3_Optimal3': ['LR', 'SVM-RBF', 'XGB'],
    'C4_Prev4':    ['LR', 'SVM-L', 'CNB', 'RF'],
    'C5_Prev5':    ['LR', 'SVM-RBF', 'CNB', 'RF', 'LGBM'],
}
ENSEMBLE_METHODS = ['Hard Voting', 'Soft Voting', 'Stacking']

n_models = len(INDIVIDUAL_MODELS) + len(ENSEMBLE_COMBOS) * len(ENSEMBLE_METHODS)
print(f"설정 완료")
print(f"  개별 모델: {len(INDIVIDUAL_MODELS)}개, 앙상블: {len(ENSEMBLE_COMBOS)}×{len(ENSEMBLE_METHODS)}={len(ENSEMBLE_COMBOS)*len(ENSEMBLE_METHODS)}개")
print(f"  총 모델: {n_models}개")
print(f"  분할: {len(SPLIT_RATIOS)} 비율 × {len(SPLIT_SEEDS)} seed = {len(SPLIT_RATIOS)*len(SPLIT_SEEDS)}개")
print(f"  총 실험: {n_models * len(SPLIT_RATIOS) * len(SPLIT_SEEDS)}회")

In [ ]:
# 데이터 로드
df = pd.read_excel(DATA_PATH)
print(f"전체 데이터: {len(df)}개, 컬럼: {df.columns.tolist()}\n")

# ID 파싱: 정규표현식으로 학생/검사지/질문 정보 추출
df['student'] = df['id'].str.extract(r'^(\d+)_')      # 학생 번호
df['form'] = df['id'].str.extract(r'_([ab])q')         # 검사지 유형
df['question'] = df['id'].str.extract(r'_[ab](q\d)_')  # 질문 번호

print("=== ID 구조 파싱 ===")
print(f"학생: {df['student'].nunique()}명, 검사지: {sorted(df['form'].unique())}, 질문: {sorted(df['question'].unique())}")

print(f"\n=== 레이블 분포 ===")
print(df['label'].value_counts().sort_index())

# 학생별 샘플 수: 학생마다 응답 수가 다름 (A+B 모두 참여 vs 한쪽만)
stu_counts = df.groupby('student').size()
print(f"\n=== 학생별 샘플 수 ===")
print(f"평균: {stu_counts.mean():.1f}, 최소: {stu_counts.min()}, 최대: {stu_counts.max()}")
# ~40개: A형+B형 모두 참여(8문항×~5응답), ~20개: 한쪽만 참여
print(f"~40개: {(stu_counts >= 30).sum()}명, ~20개: {(stu_counts < 30).sum()}명")

# 형태소 분석 (Kiwi + Lemmatization)
class KiwiLemmaTokenizer:
    INCLUDE_POS = {'NNG','NNP','NNB','NR','VV','VA','VX','MAG','MAJ'}

    def __init__(self):
        self.kiwi = Kiwi()

    def tokenize_with_lemma(self, text):
        if pd.isna(text) or str(text).strip() == '':
            return ''
        tokens = self.kiwi.tokenize(str(text))
        return ' '.join(t.lemma or t.form for t in tokens if t.tag in self.INCLUDE_POS)

print("\n토큰화 중...")
tokenizer = KiwiLemmaTokenizer()
df['tokenized'] = df['feedback_text'].apply(tokenizer.tokenize_with_lemma)
print(f"토큰화 완료! (예시: '{df['tokenized'].iloc[0]}')") 

## 2. 학생 기반 분할 함수

### 분할 및 검증 함수 정의

- `split_by_students()`: 학생 ID를 기준으로 분할합니다. 한 학생의 모든 피드백은 훈련 또는 테스트 한쪽에만 배정됩니다.
- `verify_split_overlap()`: 같은 비율의 서로 다른 시드 간 테스트 학생의 겹침 정도를 확인합니다. 겹침이 적을수록 3개 시드가 다양한 학생 구성을 평가합니다.

In [ ]:
def split_by_students(df, train_ratio, seed):
    """학생 ID 기준 분할: 한 학생의 모든 데이터는 한쪽에만 배정됨.
    
    Parameters:
        df: 전체 데이터프레임
        train_ratio: 훈련 학생 비율 (예: 80이면 80%)
        seed: 재현성을 위한 랜덤 시드
    """
    student_ids = sorted(df['student'].unique())
    n_students = len(student_ids)
    n_train = int(n_students * train_ratio / 100)  # 훈련 학생 수
    n_test = n_students - n_train                    # 테스트 학생 수
    
    rng = np.random.RandomState(seed)
    shuffled = rng.permutation(student_ids)  # 무작위로 학생 순서 섞기
    
    train_students = set(shuffled[:n_train])  # 앞 N명 → 훈련
    test_students = set(shuffled[n_train:])    # 나머지 → 테스트
    
    # 학생 기준으로 데이터 필터링
    train_df = df[df['student'].isin(train_students)].copy()
    test_df = df[df['student'].isin(test_students)].copy()
    
    return train_df, test_df, train_students, test_students


def verify_split_overlap(splits_info):
    """같은 비율의 서로 다른 시드 간 테스트 학생 겹침을 검증."""
    print("\n=== 분할 겹침 검증 ===")
    for ratio in SPLIT_RATIOS:
        ratio_name = f"{ratio[0]}:{ratio[1]}"
        test_sets = []
        for seed in SPLIT_SEEDS:
            key = f"{ratio_name}_seed{seed}"
            if key in splits_info:
                test_sets.append((seed, splits_info[key]['test_students']))
        
        if len(test_sets) >= 2:
            print(f"\n[{ratio_name}]")
            for i in range(len(test_sets)):
                for j in range(i+1, len(test_sets)):
                    s1, set1 = test_sets[i]
                    s2, set2 = test_sets[j]
                    overlap = set1 & set2  # 교집합 = 두 시드에 공통으로 포함된 테스트 학생
                    pct = len(overlap) / len(set1) * 100
                    print(f"  Seed {s1} vs {s2}: 겹침 {len(overlap)}/{len(set1)}명 ({pct:.1f}%)")

print("함수 정의 완료")

In [ ]:
# 15개 분할 생성 (5 비율 × 3 시드)
splits = {}
splits_info = {}

print("=== 학생 분할 생성 ===\n")
for train_pct, test_pct in SPLIT_RATIOS:
    ratio_name = f"{train_pct}:{test_pct}"
    for seed in SPLIT_SEEDS:
        key = f"{ratio_name}_seed{seed}"
        train_df, test_df, train_stu, test_stu = split_by_students(df, train_pct, seed)
        
        splits[key] = (train_df, test_df)
        splits_info[key] = {
            'train_students': train_stu,
            'test_students': test_stu,
            'train_n': len(train_df),
            'test_n': len(test_df),
        }
        
        print(f"[{key}] Train: {len(train_stu)}명({len(train_df)}건), Test: {len(test_stu)}명({len(test_df)}건)")

# 시드 간 테스트 학생 겹침 검증
verify_split_overlap(splits_info)

print(f"\n총 분할: {len(splits)}개")

In [ ]:
def create_vectorizer():
    """V1_Word: Word TF-IDF (lemma, ngram 1-2)"""
    return TfidfVectorizer(
        tokenizer=lambda x: x.split() if x else [],
        preprocessor=lambda x: x,
        ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
    )


def get_single_classifier(name, random_state=42):
    """9개 개별 분류기 정의."""
    clfs = {
        'LR': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=random_state, n_jobs=-1),
        'SVM-L': LinearSVC(class_weight='balanced', max_iter=2000, random_state=random_state),
        'SVM-RBF': SVC(kernel='rbf', class_weight='balanced', random_state=random_state, probability=True),
        'CNB': ComplementNB(alpha=1.0),
        'DT': DecisionTreeClassifier(class_weight='balanced', random_state=random_state),
        'RF': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=random_state, n_jobs=-1),
        'GB': GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        'XGB': XGBClassifier(n_estimators=100, eval_metric='mlogloss', random_state=random_state, n_jobs=-1, verbosity=0),
        'LGBM': LGBMClassifier(n_estimators=100, class_weight='balanced', random_state=random_state, n_jobs=-1, verbose=-1),
    }
    return clfs[name]


def get_soft_classifier(name, random_state=42):
    """Soft Voting/Stacking용: LinearSVC는 확률 출력이 불가하므로 SVC로 대체."""
    if name == 'SVM-L':
        return SVC(kernel='linear', class_weight='balanced', random_state=random_state, probability=True)
    return get_single_classifier(name, random_state)


def build_ensemble(combo_models, method, random_state=42):
    """앙상블 분류기 구성."""
    if method == 'Hard Voting':
        est = [(n.lower().replace('-','_'), get_single_classifier(n, random_state)) for n in combo_models]
        return VotingClassifier(estimators=est, voting='hard', n_jobs=-1)
    elif method == 'Soft Voting':
        est = [(n.lower().replace('-','_'), get_soft_classifier(n, random_state)) for n in combo_models]
        return VotingClassifier(estimators=est, voting='soft', n_jobs=-1)
    elif method == 'Stacking':
        est = [(n.lower().replace('-','_'), get_soft_classifier(n, random_state)) for n in combo_models]
        return StackingClassifier(
            estimators=est,
            final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=random_state),
            cv=5, n_jobs=-1
        )


def run_cv_and_predict_test(clf, X_train, y_train, X_test, n_splits=10, random_state=42):
    """Stratified 10-Fold CV + 10개 폴드 모델의 다수결 투표로 테스트 예측.
    
    일반적인 CV와 달리, 각 폴드에서 학습된 모델이 테스트 세트도 예측합니다.
    10개 모델의 예측 중 가장 많이 선택된 레이블이 최종 예측이 됩니다.
    이 방식은 단일 모델의 불안정성을 줄여주는 효과가 있습니다.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_scores = {'acc': [], 'f1': [], 'kappa': []}
    test_predictions = []  # 각 폴드 모델의 테스트 예측을 저장
    
    for tr_idx, val_idx in skf.split(X_train, y_train):
        clf_c = deepcopy(clf)
        clf_c.fit(X_train[tr_idx], y_train[tr_idx])
        
        # 검증 폴드 성능 평가 (CV 지표)
        y_val_pred = clf_c.predict(X_train[val_idx])
        cv_scores['acc'].append(accuracy_score(y_train[val_idx], y_val_pred))
        cv_scores['f1'].append(f1_score(y_train[val_idx], y_val_pred, average='macro'))
        cv_scores['kappa'].append(cohen_kappa_score(y_train[val_idx], y_val_pred))
        
        # 이 폴드 모델로 테스트 세트 예측
        test_predictions.append(clf_c.predict(X_test))
    
    # 10개 폴드 모델의 예측을 다수결 투표로 결합
    test_preds_array = np.array(test_predictions)  # shape: (10, n_test)
    final_test_pred = stats.mode(test_preds_array, axis=0, keepdims=False)[0]  # 각 샘플별 최빈값
    
    cv_result = {
        'CV_Acc': np.mean(cv_scores['acc']), 'CV_Acc_Std': np.std(cv_scores['acc']),
        'CV_F1': np.mean(cv_scores['f1']), 'CV_F1_Std': np.std(cv_scores['f1']),
        'CV_Kappa': np.mean(cv_scores['kappa']), 'CV_Kappa_Std': np.std(cv_scores['kappa']),
    }
    
    return cv_result, final_test_pred


def evaluate_test(y_true, y_pred):
    """테스트 세트 성능 평가."""
    return {
        'Test_Acc': accuracy_score(y_true, y_pred),
        'Test_F1': f1_score(y_true, y_pred, average='macro'),
        'Test_F1_W': f1_score(y_true, y_pred, average='weighted'),
        'Test_Kappa': cohen_kappa_score(y_true, y_pred),
    }


print("분류기 및 실험 함수 정의 완료")
print(f"  개별: {INDIVIDUAL_MODELS}")
print(f"  앙상블 조합: {list(ENSEMBLE_COMBOS.keys())}")
print(f"  앙상블 전략: {ENSEMBLE_METHODS}")
print(f"  테스트 예측: 10-fold 모델 다수결 투표")

In [ ]:
# 데이터 로드
df = pd.read_excel(DATA_PATH)
print(f"전체 데이터: {len(df)}개, 컬럼: {df.columns.tolist()}\n")

# ID 파싱
df['student'] = df['id'].str.extract(r'^(\d+)_')
df['form'] = df['id'].str.extract(r'_([ab])q')
df['question'] = df['id'].str.extract(r'_[ab](q\d)_')

print("=== ID 구조 파싱 ===")
print(f"학생: {df['student'].nunique()}명, 검사지: {sorted(df['form'].unique())}, 질문: {sorted(df['question'].unique())}")

print(f"\n=== 레이블 분포 ===")
print(df['label'].value_counts().sort_index())

# 학생별 샘플 수
stu_counts = df.groupby('student').size()
print(f"\n=== 학생별 샘플 수 ===")
print(f"평균: {stu_counts.mean():.1f}, 최소: {stu_counts.min()}, 최대: {stu_counts.max()}")
print(f"~40개: {(stu_counts >= 30).sum()}명, ~20개: {(stu_counts < 30).sum()}명")

# 형태소 분석 (Kiwi + Lemmatization)
class KiwiLemmaTokenizer:
    INCLUDE_POS = {'NNG','NNP','NNB','NR','VV','VA','VX','MAG','MAJ'}

    def __init__(self):
        self.kiwi = Kiwi()

    def tokenize_with_lemma(self, text):
        if pd.isna(text) or str(text).strip() == '':
            return ''
        tokens = self.kiwi.tokenize(str(text))
        return ' '.join(t.lemma or t.form for t in tokens if t.tag in self.INCLUDE_POS)

print("\n토큰화 중...")
tokenizer = KiwiLemmaTokenizer()
df['tokenized'] = df['feedback_text'].apply(tokenizer.tokenize_with_lemma)
print(f"토큰화 완료! (예시: '{df['tokenized'].iloc[0]}')") 

## 2. 학생 기반 분할 함수

In [ ]:
def split_by_students(df, train_ratio, seed):
    """Split data by student ID with given train ratio and random seed.
    
    All data from a student goes to either train or test (complete isolation).
    Uses stratified assignment considering each student's label distribution.
    """
    student_ids = sorted(df['student'].unique())
    n_students = len(student_ids)
    n_train = int(n_students * train_ratio / 100)
    n_test = n_students - n_train
    
    rng = np.random.RandomState(seed)
    shuffled = rng.permutation(student_ids)
    
    train_students = set(shuffled[:n_train])
    test_students = set(shuffled[n_train:])
    
    train_df = df[df['student'].isin(train_students)].copy()
    test_df = df[df['student'].isin(test_students)].copy()
    
    return train_df, test_df, train_students, test_students


def verify_split_overlap(splits_info):
    """Verify that different seeds produce different student assignments."""
    print("\n=== 분할 겹침 검증 ===")
    for ratio in SPLIT_RATIOS:
        ratio_name = f"{ratio[0]}:{ratio[1]}"
        test_sets = []
        for seed in SPLIT_SEEDS:
            key = f"{ratio_name}_seed{seed}"
            if key in splits_info:
                test_sets.append((seed, splits_info[key]['test_students']))
        
        if len(test_sets) >= 2:
            print(f"\n[{ratio_name}]")
            for i in range(len(test_sets)):
                for j in range(i+1, len(test_sets)):
                    s1, set1 = test_sets[i]
                    s2, set2 = test_sets[j]
                    overlap = set1 & set2
                    pct = len(overlap) / len(set1) * 100
                    print(f"  Seed {s1} vs {s2}: 겹침 {len(overlap)}/{len(set1)}명 ({pct:.1f}%)")

print("함수 정의 완료")

In [ ]:
# Generate all 15 splits (5 ratios × 3 seeds)
splits = {}
splits_info = {}

print("=== 학생 분할 생성 ===\n")
for train_pct, test_pct in SPLIT_RATIOS:
    ratio_name = f"{train_pct}:{test_pct}"
    for seed in SPLIT_SEEDS:
        key = f"{ratio_name}_seed{seed}"
        train_df, test_df, train_stu, test_stu = split_by_students(df, train_pct, seed)
        
        splits[key] = (train_df, test_df)
        splits_info[key] = {
            'train_students': train_stu,
            'test_students': test_stu,
            'train_n': len(train_df),
            'test_n': len(test_df),
        }
        
        print(f"[{key}] Train: {len(train_stu)}명({len(train_df)}건), Test: {len(test_stu)}명({len(test_df)}건)")

verify_split_overlap(splits_info)

print(f"\n총 분할: {len(splits)}개")

In [ ]:
def create_vectorizer():
    """V1_Word: Word TF-IDF (lemma, ngram 1-2)"""
    return TfidfVectorizer(
        tokenizer=lambda x: x.split() if x else [],
        preprocessor=lambda x: x,
        ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
    )


def get_single_classifier(name, random_state=42):
    """9개 개별 분류기 정의."""
    clfs = {
        'LR': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=random_state, n_jobs=-1),
        'SVM-L': LinearSVC(class_weight='balanced', max_iter=2000, random_state=random_state),
        'SVM-RBF': SVC(kernel='rbf', class_weight='balanced', random_state=random_state, probability=True),
        'CNB': ComplementNB(alpha=1.0),
        'DT': DecisionTreeClassifier(class_weight='balanced', random_state=random_state),
        'RF': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=random_state, n_jobs=-1),
        'GB': GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        'XGB': XGBClassifier(n_estimators=100, eval_metric='mlogloss', random_state=random_state, n_jobs=-1, verbosity=0),
        'LGBM': LGBMClassifier(n_estimators=100, class_weight='balanced', random_state=random_state, n_jobs=-1, verbose=-1),
    }
    return clfs[name]


def get_soft_classifier(name, random_state=42):
    """Soft Voting/Stacking용 (LinearSVC→SVC로 대체)."""
    if name == 'SVM-L':
        return SVC(kernel='linear', class_weight='balanced', random_state=random_state, probability=True)
    return get_single_classifier(name, random_state)


def build_ensemble(combo_models, method, random_state=42):
    """앙상블 분류기 구성."""
    if method == 'Hard Voting':
        est = [(n.lower().replace('-','_'), get_single_classifier(n, random_state)) for n in combo_models]
        return VotingClassifier(estimators=est, voting='hard', n_jobs=-1)
    elif method == 'Soft Voting':
        est = [(n.lower().replace('-','_'), get_soft_classifier(n, random_state)) for n in combo_models]
        return VotingClassifier(estimators=est, voting='soft', n_jobs=-1)
    elif method == 'Stacking':
        est = [(n.lower().replace('-','_'), get_soft_classifier(n, random_state)) for n in combo_models]
        return StackingClassifier(
            estimators=est,
            final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=random_state),
            cv=5, n_jobs=-1
        )


def run_cv_and_predict_test(clf, X_train, y_train, X_test, n_splits=10, random_state=42):
    """Stratified 10-Fold CV + fold model predictions on test set via majority vote."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_scores = {'acc': [], 'f1': [], 'kappa': []}
    test_predictions = []
    
    for tr_idx, val_idx in skf.split(X_train, y_train):
        clf_c = deepcopy(clf)
        clf_c.fit(X_train[tr_idx], y_train[tr_idx])
        
        y_val_pred = clf_c.predict(X_train[val_idx])
        cv_scores['acc'].append(accuracy_score(y_train[val_idx], y_val_pred))
        cv_scores['f1'].append(f1_score(y_train[val_idx], y_val_pred, average='macro'))
        cv_scores['kappa'].append(cohen_kappa_score(y_train[val_idx], y_val_pred))
        
        test_predictions.append(clf_c.predict(X_test))
    
    test_preds_array = np.array(test_predictions)
    final_test_pred = stats.mode(test_preds_array, axis=0, keepdims=False)[0]
    
    cv_result = {
        'CV_Acc': np.mean(cv_scores['acc']), 'CV_Acc_Std': np.std(cv_scores['acc']),
        'CV_F1': np.mean(cv_scores['f1']), 'CV_F1_Std': np.std(cv_scores['f1']),
        'CV_Kappa': np.mean(cv_scores['kappa']), 'CV_Kappa_Std': np.std(cv_scores['kappa']),
    }
    
    return cv_result, final_test_pred


def evaluate_test(y_true, y_pred):
    """테스트 성능 평가."""
    return {
        'Test_Acc': accuracy_score(y_true, y_pred),
        'Test_F1': f1_score(y_true, y_pred, average='macro'),
        'Test_F1_W': f1_score(y_true, y_pred, average='weighted'),
        'Test_Kappa': cohen_kappa_score(y_true, y_pred),
    }


print("분류기 및 실험 함수 정의 완료")
print(f"  개별: {INDIVIDUAL_MODELS}")
print(f"  앙상블 조합: {list(ENSEMBLE_COMBOS.keys())}")
print(f"  앙상블 전략: {ENSEMBLE_METHODS}")
print(f"  테스트 예측: 10-fold 모델 다수결 투표")

## 3. 전체 실험 실행

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(df['label'])

total_exp = len(splits) * (len(INDIVIDUAL_MODELS) + len(ENSEMBLE_COMBOS) * len(ENSEMBLE_METHODS))
all_results = []

print(f"전체 실험: {len(splits)} 분할 × 24 알고리즘 = {total_exp}회")
print(f"시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

pbar = tqdm(total=total_exp, desc="전체 진행률")

for split_name, (train_df, test_df) in splits.items():
    # Parse ratio and seed from split name
    parts = split_name.split('_seed')
    ratio_str = parts[0]
    seed_val = int(parts[1])
    train_pct = int(ratio_str.split(':')[0])
    
    info = splits_info[split_name]
    
    # Vectorize
    vectorizer = create_vectorizer()
    X_train = vectorizer.fit_transform(train_df['tokenized'])
    X_test = vectorizer.transform(test_df['tokenized'])
    y_train = label_encoder.transform(train_df['label'])
    y_test = label_encoder.transform(test_df['label'])
    
    # Predictions DataFrame
    pred_df = test_df[['id', 'feedback_text', 'label']].copy()
    pred_df = pred_df.rename(columns={'label': 'true_label'})
    
    # Individual models
    for model_name in INDIVIDUAL_MODELS:
        try:
            clf = get_single_classifier(model_name, RANDOM_SEED)
            cv_result, y_pred = run_cv_and_predict_test(clf, X_train, y_train, X_test, N_FOLDS, RANDOM_SEED)
            test_m = evaluate_test(y_test, y_pred)
            pred_df[f'{model_name}_pred'] = label_encoder.inverse_transform(y_pred)
            
            all_results.append({
                'Split': split_name, 'Ratio': ratio_str, 'Train_Pct': train_pct,
                'Seed': seed_val,
                'Train_Students': len(info['train_students']),
                'Test_Students': len(info['test_students']),
                'Train_N': info['train_n'], 'Test_N': info['test_n'],
                'Type': 'Individual', 'Model': model_name,
                **cv_result, **test_m
            })
        except Exception as e:
            print(f"Error: {split_name} {model_name}: {e}")
        pbar.update(1)
        pbar.set_postfix(split=split_name, model=model_name)
    
    # Ensemble models
    for combo_name, combo_models in ENSEMBLE_COMBOS.items():
        for method in ENSEMBLE_METHODS:
            exp_name = f"{combo_name}_{method.replace(' ', '')}"
            try:
                ens_clf = build_ensemble(combo_models, method, RANDOM_SEED)
                cv_result, y_pred = run_cv_and_predict_test(ens_clf, X_train, y_train, X_test, N_FOLDS, RANDOM_SEED)
                test_m = evaluate_test(y_test, y_pred)
                pred_df[exp_name] = label_encoder.inverse_transform(y_pred)
                
                all_results.append({
                    'Split': split_name, 'Ratio': ratio_str, 'Train_Pct': train_pct,
                    'Seed': seed_val,
                    'Train_Students': len(info['train_students']),
                    'Test_Students': len(info['test_students']),
                    'Train_N': info['train_n'], 'Test_N': info['test_n'],
                    'Type': 'Ensemble', 'Model': exp_name,
                    'Combo_Name': combo_name, 'Method': method,
                    **cv_result, **test_m
                })
            except Exception as e:
                print(f"Error: {split_name} {exp_name}: {e}")
            pbar.update(1)
    
    # Save predictions
    safe_name = split_name.replace(':', '_')
    pred_df.to_excel(PRED_DIR / f'predictions_{safe_name}.xlsx', index=False)

pbar.close()
print(f"\n실험 완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 4. 결과 저장

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_DIR / 'all_results_student_split.csv', index=False)
print(f"결과 저장: {OUTPUT_DIR / 'all_results_student_split.csv'} ({results_df.shape})")

# Summary by ratio
print(f"\n=== 비율별 최고 성능 (Test Kappa) ===")
for ratio_str in [f"{r[0]}:{r[1]}" for r in SPLIT_RATIOS]:
    subset = results_df[results_df['Ratio'] == ratio_str]
    if len(subset) > 0:
        best = subset.loc[subset['Test_Kappa'].idxmax()]
        print(f"  {ratio_str}: {best['Model']:30s} Seed={best['Seed']} Kappa={best['Test_Kappa']:.4f} F1={best['Test_F1']:.4f}")

# Summary by ratio (averaged across seeds)
print(f"\n=== 비율별 평균 성능 (3 seed 평균) ===")
avg_by_ratio = results_df.groupby('Ratio').agg({
    'Test_Kappa': 'mean', 'Test_F1': 'mean', 'CV_Kappa': 'mean', 'CV_F1': 'mean'
}).round(4)
print(avg_by_ratio)

# Save config
config = {
    'Date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'Experiment': 'Student Split ML',
    'Split_Ratios': SPLIT_RATIOS,
    'Seeds': SPLIT_SEEDS,
    'Total_Splits': len(splits),
    'Total_Experiments': len(all_results),
    'Random_Seed': RANDOM_SEED,
    'CV_Folds': N_FOLDS,
    'Test_Prediction_Method': 'Majority voting from 10-fold CV models',
}
with open(OUTPUT_DIR / 'experiment_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"\n완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")